In [5]:
import pandas as pd
import numpy as np
import torch

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score
)

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)

print("Libraries loaded successfully!")
print("PyTorch:", torch.__version__)
print("Transformers:", __import__("transformers").__version__)
print("CUDA available:", torch.cuda.is_available())

c:\Users\shind\OneDrive\Documents\Desktop\TY\EcommerceConsumerIntelligence\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Libraries loaded successfully!
PyTorch: 2.13.0+cpu
Transformers: 5.16.1
CUDA available: False


In [6]:
home_df = pd.read_csv("../data/processed/home_kitchen_balanced.csv")

print("Dataset shape:", home_df.shape)

print("\nColumns:")
print(home_df.columns.tolist())

print("\nSentiment distribution:")
print(home_df["sentiment"].value_counts())

Dataset shape: (15000, 11)

Columns:
['rating', 'title', 'text', 'images', 'asin', 'parent_asin', 'user_id', 'timestamp', 'helpful_vote', 'verified_purchase', 'sentiment']

Sentiment distribution:
sentiment
positive    5000
neutral     5000
negative    5000
Name: count, dtype: int64


In [ ]:
#convert sentiment labels to numerical values
label_map = {
    "negative": 0,
    "neutral": 1,
    "positive": 2
}

home_df["label"] = home_df["sentiment"].map(label_map)

print(home_df[["sentiment", "label"]].head(10))

print("\nLabel distribution:")
print(home_df["label"].value_counts().sort_index())

  sentiment  label
0  positive      2
1   neutral      1
2  positive      2
3  negative      0
4   neutral      1
5   neutral      1
6  negative      0
7  negative      0
8  negative      0
9  negative      0

Label distribution:
label
0    5000
1    5000
2    5000
Name: count, dtype: int64


In [ ]:
#reviews preparation for model training
home_model_df = home_df[["text", "label"]].copy()

home_model_df["text"] = (
    home_model_df["text"]
    .fillna("")
    .astype(str)
)

home_model_df = home_model_df[
    home_model_df["text"].str.strip() != ""
].reset_index(drop=True)

print("Model dataset shape:", home_model_df.shape)

print("\nSample reviews:")
print(home_model_df.head())

print("\nLabel distribution:")
print(home_model_df["label"].value_counts().sort_index())

Model dataset shape: (14997, 2)

Sample reviews:
                                                text  label
0  Works great!  I have to use the tiny kernel po...      2
1  I’ve ordered 4 different down comforters throu...      1
2  Eco friendly fillable tea bag with good  draw ...      2
3  The wind blowing by my ears gave me a really b...      0
4  This mirror feels cheap. It has a cheap plasti...      1

Label distribution:
label
0    5000
1    4998
2    4999
Name: count, dtype: int64


In [ ]:
#training and testing data split
X = home_model_df["text"]
y = home_model_df["label"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))

print("\nTraining label distribution:")
print(y_train.value_counts().sort_index())

print("\nTesting label distribution:")
print(y_test.value_counts().sort_index())

Training samples: 11997
Testing samples: 3000

Training label distribution:
label
0    4000
1    3998
2    3999
Name: count, dtype: int64

Testing label distribution:
label
0    1000
1    1000
2    1000
Name: count, dtype: int64


In [ ]:
#loading the tokenizer and model
model_name = "distilbert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(model_name)

print("Tokenizer loaded successfully!")
print("Model:", model_name)

Tokenizer loaded successfully!
Model: distilbert-base-uncased


In [ ]:
#tokenizing the reviews
train_encodings = tokenizer(
    X_train.tolist(),
    truncation=True,
    padding=True,
    max_length=128
)

test_encodings = tokenizer(
    X_test.tolist(),
    truncation=True,
    padding=True,
    max_length=128
)

print("Tokenization completed!")
print("Training reviews:", len(train_encodings["input_ids"]))
print("Testing reviews:", len(test_encodings["input_ids"]))
print("Tokens in first review:", len(train_encodings["input_ids"][0]))

Tokenization completed!
Training reviews: 11997
Testing reviews: 3000
Tokens in first review: 128


In [ ]:
#pytorch dataset class for Amazon reviews
class AmazonReviewDataset(torch.utils.data.Dataset):

    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):

        item = {
            key: torch.tensor(val[idx])
            for key, val in self.encodings.items()
        }

        item["labels"] = torch.tensor(self.labels[idx])

        return item

    def __len__(self):
        return len(self.labels)


train_dataset = AmazonReviewDataset(
    train_encodings,
    y_train.tolist()
)

test_dataset = AmazonReviewDataset(
    test_encodings,
    y_test.tolist()
)

print("PyTorch datasets created!")
print("Training dataset:", len(train_dataset))
print("Testing dataset:", len(test_dataset))

PyTorch datasets created!
Training dataset: 11997
Testing dataset: 3000


In [ ]:
#creating the DistilBERT model for sequence classification
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=3
)

print("DistilBERT model loaded successfully!")
print("Number of labels:", model.config.num_labels)

Loading weights: 100%|██████████| 100/100 [00:00<00:00, 1241.64it/s]
[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


DistilBERT model loaded successfully!
Number of labels: 3


In [1]:
def compute_metrics(eval_pred):

    predictions, labels = eval_pred

    predictions = np.argmax(predictions, axis=1)

    accuracy = accuracy_score(
        labels,
        predictions
    )

    macro_f1 = f1_score(
        labels,
        predictions,
        average="macro"
    )

    return {
        "accuracy": accuracy,
        "macro_f1": macro_f1
    }


print("Evaluation metrics defined successfully!")

Evaluation metrics defined successfully!


In [4]:
training_args = TrainingArguments(
    output_dir="../outputs/transformer_home",
    eval_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=1,
    weight_decay=0.01,
    logging_steps=100,
    save_strategy="no",
    report_to="none"
)

print("Training configuration created!")

Training configuration created!


In [12]:
trainer = Trainer(

    model=model,

    args=training_args,

    train_dataset=train_dataset,

    eval_dataset=test_dataset,

    compute_metrics=compute_metrics
)

print("Trainer created successfully!")

Trainer created successfully!


In [3]:
train_result = trainer.train()

print("Transformer training completed!")

NameError: name 'trainer' is not defined

In [4]:
required_variables = [
    "model",
    "training_args",
    "train_dataset",
    "test_dataset",
    "compute_metrics",
    "trainer"
]

for name in required_variables:
    print(f"{name}: {name in globals()}")

model: False
training_args: False
train_dataset: False
test_dataset: False
compute_metrics: False
trainer: False


In [3]:
import transformers
from transformers import TrainingArguments, Trainer

print("Transformers version:", transformers.__version__)
print("TrainingArguments:", TrainingArguments)
print("Trainer:", Trainer)

c:\Users\shind\OneDrive\Documents\Desktop\TY\EcommerceConsumerIntelligence\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Transformers version: 5.16.1
TrainingArguments: <class 'transformers.training_args.TrainingArguments'>
Trainer: <class 'transformers.trainer.Trainer'>


In [5]:
required_variables = [
    "model",
    "train_dataset",
    "test_dataset",
    "compute_metrics",
    "training_args"
]

for name in required_variables:
    print(f"{name}: {name in globals()}")

model: False
train_dataset: False
test_dataset: False
compute_metrics: True
training_args: True


In [6]:
import torch

class AmazonReviewDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {
            key: torch.tensor(val[idx])
            for key, val in self.encodings.items()
        }
        item["labels"] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)


train_dataset = AmazonReviewDataset(
    train_encodings,
    y_train.tolist()
)

test_dataset = AmazonReviewDataset(
    test_encodings,
    y_test.tolist()
)

print("Training dataset:", len(train_dataset))
print("Testing dataset:", len(test_dataset))

NameError: name 'train_encodings' is not defined

In [8]:
import pandas as pd
import numpy as np
import torch

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)

home_df = pd.read_csv(
    "../data/processed/home_kitchen_balanced.csv"
)

print("Dataset loaded successfully!")
print("Shape:", home_df.shape)
print(home_df.head())

Dataset loaded successfully!
Shape: (15000, 11)
   rating                                              title  \
0     5.0                            silicone popcorn popper   
1     3.0                        Not the best I’ve purchased   
2     5.0                   Perfect for loose tea or coffee!   
3     1.0                                    Ear ache Alert!   
4     3.0  Cheaply designed mirror with one brightness se...   

                                                text  \
0  Works great!  I have to use the tiny kernel po...   
1  I’ve ordered 4 different down comforters throu...   
2  Eco friendly fillable tea bag with good  draw ...   
3  The wind blowing by my ears gave me a really b...   
4  This mirror feels cheap. It has a cheap plasti...   

                                              images        asin parent_asin  \
0                                                 []  B07JW23NG4  B09VVQK1ZT   
1                                                 []  B07KY7W18K  B099

In [9]:
label_map = {
    "negative": 0,
    "neutral": 1,
    "positive": 2
}

home_df["label"] = home_df["sentiment"].map(label_map)

home_model_df = home_df[["text", "label"]].copy()

home_model_df["text"] = (
    home_model_df["text"]
    .fillna("")
    .astype(str)
)

home_model_df = home_model_df[
    home_model_df["text"].str.strip() != ""
].reset_index(drop=True)

print("Prepared dataset:", home_model_df.shape)
print(home_model_df["label"].value_counts().sort_index())

Prepared dataset: (14997, 2)
label
0    5000
1    4998
2    4999
Name: count, dtype: int64


In [ ]:
#splitting the dataset into training and testing sets
X = home_model_df["text"]
y = home_model_df["label"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training reviews:", len(X_train))
print("Testing reviews:", len(X_test))

Training reviews: 11997
Testing reviews: 3000


In [ ]:
#loading the tokenizer and model
model_name = "distilbert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(model_name)

print("Tokenizer loaded successfully!")

Tokenizer loaded successfully!


In [12]:
train_encodings = tokenizer(
    X_train.tolist(),
    truncation=True,
    padding=True,
    max_length=128
)

test_encodings = tokenizer(
    X_test.tolist(),
    truncation=True,
    padding=True,
    max_length=128
)

print("Tokenization completed!")
print("Training reviews:", len(train_encodings["input_ids"]))
print("Testing reviews:", len(test_encodings["input_ids"]))

Tokenization completed!
Training reviews: 11997
Testing reviews: 3000


In [13]:
class AmazonReviewDataset(torch.utils.data.Dataset):

    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {
            key: torch.tensor(val[idx])
            for key, val in self.encodings.items()
        }
        item["labels"] = torch.tensor(
            self.labels[idx], dtype=torch.long
        )
        return item

    def __len__(self):
        return len(self.labels)


train_dataset = AmazonReviewDataset(
    train_encodings,
    y_train.tolist()
)

test_dataset = AmazonReviewDataset(
    test_encodings,
    y_test.tolist()
)

print("Training dataset:", len(train_dataset))
print("Testing dataset:", len(test_dataset))

Training dataset: 11997
Testing dataset: 3000


In [14]:
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=3,
    id2label={
        0: "negative",
        1: "neutral",
        2: "positive"
    },
    label2id={
        "negative": 0,
        "neutral": 1,
        "positive": 2
    }
)

print("DistilBERT model loaded!")
print("Number of labels:", model.config.num_labels)

Loading weights: 100%|██████████| 100/100 [00:00<00:00, 2486.97it/s]
[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


DistilBERT model loaded!
Number of labels: 3


In [15]:
def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)

    return {
        "accuracy": accuracy_score(labels, predictions),
        "macro_f1": f1_score(
            labels, predictions, average="macro"
        )
    }

print("Evaluation metrics ready!")

Evaluation metrics ready!


In [16]:
training_args = TrainingArguments(
    output_dir="../outputs/transformer_home",
    eval_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=1,
    weight_decay=0.01,
    logging_steps=100,
    save_strategy="epoch",
    report_to="none"
)

print("Training configuration ready!")

Training configuration ready!


In [17]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics
)

print("Trainer created successfully!")

Trainer created successfully!


In [18]:
for name in [
    "model",
    "train_dataset",
    "test_dataset",
    "compute_metrics",
    "training_args",
    "trainer"
]:
    print(f"{name}: {name in globals()}")

model: True
train_dataset: True
test_dataset: True
compute_metrics: True
training_args: True
trainer: True


In [19]:
train_result = trainer.train()

print("Transformer training completed!")

c:\Users\shind\OneDrive\Documents\Desktop\TY\EcommerceConsumerIntelligence\.venv\Lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss,Accuracy,Macro F1
1,0.557971,0.594987,0.740333,0.741090


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  2.66it/s]


Transformer training completed!


In [20]:
eval_results = trainer.evaluate()

print("Accuracy:", round(eval_results["eval_accuracy"], 4))
print("Macro F1:", round(eval_results["eval_macro_f1"], 4))
print("Evaluation loss:", round(eval_results["eval_loss"], 4))

c:\Users\shind\OneDrive\Documents\Desktop\TY\EcommerceConsumerIntelligence\.venv\Lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Training Loss,Validation Loss,Epoch,Accuracy,Macro F1
0.557971,0.594987,1,0.740333,0.741090


Accuracy: 0.7403
Macro F1: 0.7411
Evaluation loss: 0.595


In [21]:
from pathlib import Path

MODEL_PATH = Path("../models/sentiment_model")
MODEL_PATH.mkdir(parents=True, exist_ok=True)

trainer.save_model(str(MODEL_PATH))
tokenizer.save_pretrained(str(MODEL_PATH))

print("Model saved to:", MODEL_PATH.resolve())

Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  5.12it/s]


Model saved to: C:\Users\shind\OneDrive\Documents\Desktop\TY\EcommerceConsumerIntelligence\models\sentiment_model


In [22]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

tokenizer_saved = AutoTokenizer.from_pretrained(str(MODEL_PATH))
model_saved = AutoModelForSequenceClassification.from_pretrained(str(MODEL_PATH))

print("Tokenizer loaded successfully!")
print("Model loaded successfully!")
print("Number of labels:", model_saved.config.num_labels)
print("Labels:", model_saved.config.id2label)

Loading weights: 100%|██████████| 104/104 [00:00<00:00, 2364.74it/s]

Tokenizer loaded successfully!
Model loaded successfully!
Number of labels: 3
Labels: {0: 'negative', 1: 'neutral', 2: 'positive'}


In [23]:
import torch

sample_review = "The product is excellent and works exactly as expected."

inputs = tokenizer_saved(
    sample_review,
    return_tensors="pt",
    truncation=True,
    padding=True,
    max_length=128
)

model_saved.eval()

with torch.no_grad():
    outputs = model_saved(**inputs)
    probabilities = torch.softmax(outputs.logits, dim=1)
    predicted_id = torch.argmax(probabilities, dim=1).item()

id_to_label = {0: "negative", 1: "neutral", 2: "positive"}

print("Review:", sample_review)
print("Predicted sentiment:", id_to_label[predicted_id])
print("Confidence:", round(probabilities[0][predicted_id].item() * 100, 2), "%")

Review: The product is excellent and works exactly as expected.
Predicted sentiment: positive
Confidence: 94.61 %


In [24]:
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix

pred_output = trainer.predict(test_dataset)

y_pred = np.argmax(pred_output.predictions, axis=1)
y_true = np.array(y_test)

print("Classification Report:\n")
print(
    classification_report(
        y_true,
        y_pred,
        labels=[0, 1, 2],
        target_names=["Negative", "Neutral", "Positive"],
        digits=4
    )
)

print("Confusion Matrix:\n")
print(confusion_matrix(y_true, y_pred, labels=[0, 1, 2]))

c:\Users\shind\OneDrive\Documents\Desktop\TY\EcommerceConsumerIntelligence\.venv\Lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Classification Report:

              precision    recall  f1-score   support

    Negative     0.7411    0.7100    0.7252      1000
     Neutral     0.6317    0.6620    0.6465      1000
    Positive     0.8541    0.8490    0.8516      1000

    accuracy                         0.7403      3000
   macro avg     0.7423    0.7403    0.7411      3000
weighted avg     0.7423    0.7403    0.7411      3000

Confusion Matrix:

[[710 261  29]
 [222 662 116]
 [ 26 125 849]]


In [25]:
from pathlib import Path
import pandas as pd
from sklearn.metrics import classification_report, confusion_matrix
import numpy as np

RESULTS_PATH = Path("../outputs/transformer_home")
RESULTS_PATH.mkdir(parents=True, exist_ok=True)

# Classification report
report = classification_report(
    y_true,
    y_pred,
    labels=[0, 1, 2],
    target_names=["Negative", "Neutral", "Positive"],
    output_dict=True,
    zero_division=0
)

pd.DataFrame(report).transpose().to_csv(
    RESULTS_PATH / "classification_report.csv"
)

# Confusion matrix
cm = confusion_matrix(y_true, y_pred, labels=[0, 1, 2])

pd.DataFrame(
    cm,
    index=["Actual Negative", "Actual Neutral", "Actual Positive"],
    columns=["Predicted Negative", "Predicted Neutral", "Predicted Positive"]
).to_csv(RESULTS_PATH / "confusion_matrix.csv")

print("Classification report saved.")
print("Confusion matrix saved.")
print("Location:", RESULTS_PATH.resolve())

Classification report saved.
Confusion matrix saved.
Location: C:\Users\shind\OneDrive\Documents\Desktop\TY\EcommerceConsumerIntelligence\outputs\transformer_home
